* openai nécessite un plan de facturation pour utiliser une API key
* mistral nécessite une API key, mais il existe un plan gratuit avec 100k tokens par mois
* anthropic nécessite une API key, mais il existe un plan gratuit avec 100k tokens par mois
* google_generative_ai nécessite une API key, mais il existe un plan gratuit avec 100k tokens par mois

In [ ]:
# choix du fournisseur d'ia
provider = "mistral" # openai, mistral, anthropic, google_generative_ai

openai nécessite un plan de facturation pour utiliser une API key
mistral nécessite une API key, mais il existe un plan gratuit avec 100k tokens par mois
anthropic nécessite une API key, mais il existe un plan gratuit avec 100k tokens par mois
google_generative_ai nécessite une API key, mais il existe un plan gratuit avec 100k tokens par mois


In [ ]:
# affiche les versions installées de python, langchain et l'executable utilisé

import sys
print(sys.version)

import langchain
print(langchain.__version__)

import sys
print(sys.executable)

3.11.15 | packaged by Anaconda, Inc. | (main, Mar 11 2026, 17:12:15) [MSC v.1942 64 bit (AMD64)]
1.2.13
c:\Users\aceteam\anaconda3\envs\ai-workflow\python.exe


In [ ]:
# vérifie la présence des API key
import os

# autres intégrations
# https://docs.langchain.com/oss/python/integrations/chat/anthropic
# https://docs.langchain.com/oss/python/integrations/chat/google_generative_ai
# https://docs.langchain.com/oss/python/integrations/chat/mistralai


# Assure-toi d'avoir ton API key
if provider == "openai" and not os.environ["OPENAI_API_KEY"]:
    raise ValueError("Please set your OpenAI API key in the environment variable OPENAI_API_KEY")
if provider == "mistral" and not os.environ["MISTRAL_API_KEY"]:
    raise ValueError("Please set your OpenAI API key in the environment variable MISTRAL_API_KEY")

In [ ]:
# fonction: parse une chaine
def parse_string(template, variables):
    # Regex pour trouver les zones de code entre backticks
    pattern = r"`([^`]+)`"

    def replace_code(match):
        code = match.group(1)  # extrait le contenu sans les backticks
        try:
            # évalue le template avec str.format et les variables
            return code.format(**variables)
        except KeyError as e:
            return code  # si variable non définie, on garde le code tel quel
    
    return re.sub(pattern, replace_code, template)

In [ ]:
# fonction: extrait les données d'un jalon dans les variables globales
def read_jalon(start_line, end_line, numero, data):
    print(f"Lecture du jalon {numero} (lignes {start_line} à {end_line})")
    """
    Lit un bloc Jalon et extrait les informations du tableau Markdown.
    Remplit un dictionnaire create_jalon() et l'ajoute à workflow_data['jalons']
    """

    jalon = create_jalon(numero)

    headers = None

    # parcours des lignes du bloc
    for i in range(start_line, end_line):
        line = lines[i].strip()

        # ignorer les lignes vides
        if not line:
            continue

        # détecte les lignes du tableau
        if '|' in line:
            # découpe les colonnes et supprime les espaces
            row = [c.strip() for c in line.strip('|').split('|')]

            # première ligne du tableau = headers
            if headers is None:
                headers = row
                continue

            # ignorer la ligne de séparateurs --- | --- | ---
            if all(re.fullmatch(r'-+', c) for c in row):
                continue

            # remplir le jalon avec les colonnes présentes dans create_jalon
            task = create_task()
            for h, v in zip(headers, row):
                key = h.strip()
                if key in task:
                    task[key] = v

            jalon.append(task)

    data["jalons"].append(jalon)

In [ ]:
# fonction: lit la totalité du document est extrait les informations dans les varaibles globales
import re

def read_workflow(start_line, end_line, data):
    """
    Cherche les sous-titres Jalon X dans les titres extraits et appelle read_jalon.
    """
    
    # pattern pour détecter Jalon 1, Jalon 2, ...
    jalon_pattern = re.compile(r'jalon\s*(\d+)', re.IGNORECASE)

    # filtrer les titres situés dans la section workflow
    workflow_titles = [t for t in titles if start_line <= t['line'] < end_line]

    # récupère les varaibles globales du workflow
    vars = {}
    index = next((i for i, t in enumerate(titles) if t['title'] == "Variables"), None)
    if index is not None:
        start = int(titles[index]["line"])
        end = int(titles[index+1]["line"]) if index+1 < len(titles) else len(lines)
        slice_lines = lines[start:end]

        # cherche le début du bloc python
        vars_start_rel = next((i for i, l in enumerate(slice_lines) if l.strip().startswith("```python")), None)
        if vars_start_rel is not None:
            vars_start = start + vars_start_rel  # indice absolu dans lines

            # cherche la fin du bloc
            vars_end_rel = next((i for i, l in enumerate(slice_lines[vars_start_rel+1:], start=vars_start_rel+1) if l.strip().startswith("```")), None)
            if vars_end_rel is not None:
                vars_end = start + vars_end_rel  # indice absolu dans lines

                # récupère le code entre ```python et ```
                vars_code = "\n".join(lines[vars_start+1:vars_end])

                # exécute le code
                exec(vars_code, {}, vars)
                data["variables"] = vars
                print("variables globales :", vars)

    # ne garder que les titres qui sont des jalons
    jalons = []
    for t in workflow_titles:
        m = jalon_pattern.match(t['title'])
        if m:
            jalons.append({
                "line": t['line'],
                "numero": int(m.group(1))
            })

    # traiter chaque jalon
    for idx, j in enumerate(jalons):
        start = j["line"] + 1
        if idx + 1 < len(jalons):
            end = jalons[idx + 1]["line"]
        else:
            end = end_line

        read_jalon(start, end, j["numero"], data)

In [ ]:
# fonction: teste la condition de validation d'un job
def check_job_validation(job, iteration, vars):
    allowed_globals = {
        "fileExists": lambda filename : os.path.isfile("produces/"+filename),
        #,"multiply": multiply
    }

    local_vars = vars | {"i": iteration}
    
    if job["Validation"] is not None:
        content = job["Validation"].strip()
        if content.startswith("`"):
            code = content.strip("`").strip()
            result = eval(code, allowed_globals, local_vars)
        else: 
            result =  False
            
        print("Validation content:", content, result, "with", local_vars)

        return result
    else:
        print(f"Job {job['Job']} du jalon {job['Jalon']} n'a pas de validation.")
        return True

In [ ]:
# fonction: crée la base de données vectorielle avec les documents déjà produits
from langchain_mistralai import ChatMistralAI, MistralAIEmbeddings
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.document_loaders import CSVLoader, UnstructuredMarkdownLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
import glob
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
import os

database_filename = "vector_db"

# modèle LLM utilisé
# les embeddings sont spécifiques au modèle utilisé pour les créer
llm = None

if provider == "mistral":
    llm = ChatMistralAI(model="mistral-large-latest")
if provider == "openai":
    llm = ChatOpenAI(model="gpt-4o-mini")

# Créer les embeddings
# Les embeddings permettent de comparer la similarité sémantique
# les embeddings sont spécifiques au modèle utilisé pour les créer
embeddings = None

if provider == "mistral":
    embeddings = MistralAIEmbeddings()
if provider == "openai":
    embeddings = OpenAIEmbeddings()

if not llm or not embeddings:
    raise ValueError("implémentation manquante pour le modèle choisi: " + provider)

# crée la base de données contextuelle
def create_database():
    docs = []

    # charge les documents du contexte
    # Chaque fichier devient un Document LangChain
    for file in glob.glob("produces/*.txt"):
        loader = TextLoader(file)
        docs.extend(loader.load())
        #docs[-1].metadata["acteur"] = job["Acteur"]
        
    for file in glob.glob("produces/*.md"):
        loader = UnstructuredMarkdownLoader(file, mode="elements")
        docs.extend(loader.load())
        #docs[-1].metadata["acteur"] = job["Acteur"]

    for file in glob.glob("produces/*.csv"):
        loader = CSVLoader(file, mode="elements")
        docs.extend(loader.load())
        #docs[-1].metadata["acteur"] = job["Acteur"]

    for doc in docs:
        doc.metadata["name"] = os.path.basename(doc.metadata["source"])


    # retourne une base de données vide
    if not docs:
        dummy = [Document(page_content="init")]
        db = FAISS.from_documents(dummy, embeddings)
        db.delete([list(db.docstore._dict.keys())[0]])
        return db

    # découpe les documents en chunks pour les fournir à l'agent
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
    #tool_docs = text_splitter.create_documents([d.page_content for d in docs])
    chunks = text_splitter.split_documents(docs)

    # Crée la base vectorielle
    # la base vectoriel est spécifique au modèle utilisé pour les embeddings
    db = FAISS.from_documents(chunks, embeddings)
    
    # Sauvegarde la base vectorielle
    db.save_local(database_filename)

    return db

# charge la base de données contextuelle
def load_database():
    # Sauvegarde la base vectorielle
    db = FAISS.load_local(
        database_filename,
        embeddings,
        allow_dangerous_deserialization=True
        )
    return db

La fonction `execute_job` inbtéroge le LLM en passant à l'agent les outils et le contexte donné en argument. 

La réponse et une discution, les éléments produits sont généralement des fichiers nommé explicitement par le Job

In [ ]:
# fonction: execute un job

from langchain.agents import create_agent
from datetime import datetime
import importlib
import os

# evite les injection de path pour les outils
def is_safe_tool_name(name):
    return re.fullmatch(r"[a-zA-Z0-9_\-]+", name) is not None

# exécute un job
def execute_job(job, iteration, vars):

    local_vars = vars | {"i": iteration}

    Etape = parse_string(job["Etape"], local_vars)
    Produit = parse_string(job["Produit"], local_vars)
    Acteur = parse_string(job['Acteur'], local_vars)
    Contexte = parse_string(job['Contexte'], local_vars)
    Outils = parse_string(job['Outils'], local_vars)

    tools = []
    
    for tool in re.split(r"[;,|\s]", Outils):
        if not is_safe_tool_name(tool):
            raise ValueError("not safe tool name:", tool)
        
        filename = "tools/"+tool+".py"
        if not os.path.isfile(filename):
            raise ValueError("tool file not found:", filename)
        
        modulename = "tools."+tool

        # importer le module (cache initial)
        module = importlib.import_module(modulename)
                
        # recharger le module pour forcer la mise à jour
        module = importlib.reload(module)

        value = getattr(module, "tools", None)
        if value and isinstance(value, list):
            tools.extend(value)
        else:
            raise ValueError("no tools list found in module:", modulename)

        print("add tool:", tool)

    print("tools:", tools)

    db = create_database()
    
    # Créer le retriever
    # (récupérer k passages pertinents par contexte)
    docs = []

    for src in re.split(r"[;,|\s]", Contexte):
        retriever = db.as_retriever(
            search_kwargs={"k": 2, "filter": {"filename": src}}
        )
        docs.extend(retriever.invoke(job["Job"]))

    print("docs:", docs)

    # Trie les résultats par score
    docs = sorted(docs, key=lambda d: d.metadata.get("score", 0), reverse=True)

    # extrait les passages pertinants pour le contexte du prompt
    prompt_context = ""
    for d in docs:
        print(d)
        prompt_context += f"Document: {d.metadata['name']}\nContent: {d.page_content}\n\n"
    
    # Créer un agent
    # permet d'utiliser les aller-retours avec des outils et skills
    agent = create_agent(llm, tools=tools)

    prompt_inputs = {
        "messages": [
            {"role": "system", "content": f"""
Tu es {Acteur} et tu dois réaliser ce travail: {Etape}.

Voici le contexte pertinent :
{prompt_context}
"""},
            {"role": "user", "content": f"Ce travail doit produire exactement ce résultat: {Produit}"}
        ]
    }

    print("prompt_inputs:", prompt_inputs)

    response = agent.invoke(prompt_inputs)

    print("response:", response)

    with open("journal.md", "a", encoding="utf-8") as f:
        f.write(f"\n--- {datetime.now()} ---\n")
        f.write(f"job: {job['Job']}\n")
        f.write("\n\n")

        for msg in response["messages"]:
            f.write(msg.__class__.__name__ + ":\n")
            f.write(msg.content + "\n")

    return True 

In [393]:
lines = load_md("project.md")

titles = extract_titles(lines)

print(titles)

workflow_data = {
    "jalons": [],
    "variables": {}
}

read_workflow(48,139, workflow_data)

# execute les jobs un à un 
for jalon in workflow_data["jalons"]:
    for job in jalon:
        i = 0
        j = int(parse_string(job["Itération"], workflow_data["variables"] | {"i": i}))

        for i in range(j):
            if not check_job_validation(job, i, workflow_data["variables"]):
                print(f"{job['Job']} => Execution...")
                result = execute_job(job, i, workflow_data["variables"])
                if result:
                    print(f"=> Validé")

                    reponse = input("Voulez-vous continuer ? (o/n) : ")
                    if reponse.lower() in ["o", "oui", "y", "yes"]:
                        print("On continue !")
                    else:
                        raise ValueError("Opération annulée.")
                else:
                    raise ValueError(f"=> Échec de l'exécution. {result}")
            else:
                print(f"{job['Job']} => OK")

[{'level': 1, 'title': '2D Platform Game', 'line': 0}, {'level': 2, 'title': 'Définition', 'line': 2}, {'level': 3, 'title': 'Le sujet', 'line': 4}, {'level': 3, 'title': 'Les objectifs', 'line': 10}, {'level': 3, 'title': 'Les parties prenantes (acteurs):', 'line': 18}, {'level': 2, 'title': 'Jalons', 'line': 28}, {'level': 2, 'title': 'Workflow', 'line': 48}, {'level': 3, 'title': 'Variables', 'line': 50}, {'level': 3, 'title': 'Jalon  1', 'line': 89}, {'level': 3, 'title': 'Jalon  2', 'line': 98}, {'level': 3, 'title': 'Jalon  3', 'line': 108}, {'level': 3, 'title': 'Jalon  4', 'line': 118}, {'level': 2, 'title': 'Skills', 'line': 139}, {'level': 3, 'title': 'Unity Coding', 'line': 141}, {'level': 3, 'title': 'UI Design', 'line': 145}, {'level': 3, 'title': 'Level design', 'line': 149}, {'level': 3, 'title': 'Level scripting', 'line': 153}, {'level': 2, 'title': 'Tools', 'line': 157}, {'level': 3, 'title': 'generate_sprite', 'line': 159}, {'level': 1, 'title': 'Mise en place des age

Retrying langchain_mistralai.chat_models.ChatMistralAI.completion_with_retry.<locals>._completion_with_retry in 4 seconds as it raised ReadTimeout: The read operation timed out.


response: {'messages': [SystemMessage(content='\nTu es Scénariste et tu dois réaliser ce travail: Décrire pièges, environnement, objectif.\n\nVoici le contexte pertinent :\nDocument: contexte.md\nContent: L’époque est marquée par une quatrième révolution industrielle, où l’intelligence artificielle, la biotechnologie et les nanotechnologies ont fusionné pour redéfinir les limites de l’humain. Les frontières entre le physique, le numérique et le biologique s’estompent : les implants neuronaux permettent de communiquer par la pensée, les corps peuvent être augmentés pour résister aux conditions extrêmes, et les villes sont gérées par des IA centralisées, garantissant une efficacité optimale mais soulevant des questions éthiques.\n\nDocument: contexte.md\nContent: Sur le plan géopolitique, le monde est divisé entre des blocs régionaux collaborant pour gérer les ressources devenues rares, comme l’eau potable et les terres arables. Les océans, acidifiés et montants, ont englouti des régions

ValueError: ('tool file not found:', 'tools/sketch.py')